<a href="https://colab.research.google.com/github/kauazin05/CIAO_KKG_2026/blob/main/AULA_07/lab02_aula07_ciao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# LAB 02 - ALGORITMO GENETICO: Selecao, Crossover e Mutacao
# Tema: Otimizacao Combinatoria (Problema da Mochila)
# ==========================================================
import numpy as np

np.random.seed(42)  # reprodutibilidade (mesma logica do LAB 01)

# Problema da Mochila (Blindagem de Ativos)
weights = np.array([12, 2, 1, 4, 1])   # Custo/Consumo de Memoria dos ativos
values  = np.array([4, 2, 1, 10, 2])   # Cobertura de Risco/Valor do ativo
max_weight = 15

pop_size = 10
num_genes = len(weights)
generations = 10
mutation_rate = 0.1

# Inicializacao da Populacao (Matriz Binaria 10x5)
population = np.random.randint(0, 2, size=(pop_size, num_genes))


def calculate_fitness(ind):
    total_weight = np.sum(ind * weights)
    total_value = np.sum(ind * values)
    # TODO 1 RESOLVIDO: restricao de peso -> penalizacao
    if total_weight > max_weight:
        return 0
    return total_value


def tournament_selection(pop, fitnesses):
    # TODO 2 RESOLVIDO: Selecao por Torneio (k = 2)
    i, j = np.random.randint(0, len(pop), size=2)
    if fitnesses[i] >= fitnesses[j]:
        return pop[i].copy()
    return pop[j].copy()


def crossover(parent1, parent2):
    point = np.random.randint(1, num_genes)
    child1 = np.concatenate([parent1[:point], parent2[point:]])
    child2 = np.concatenate([parent2[:point], parent1[point:]])
    return child1, child2


def mutate(ind):
    for i in range(num_genes):
        if np.random.rand() < mutation_rate:
            ind[i] = 1 - ind[i]   # Inverte o bit (0 -> 1 ou 1 -> 0)
    return ind


# Historico para o grafico de convergencia
convergence_best = []   # melhor fitness da geracao
convergence_mean = []   # fitness medio da geracao
best_ind, best_fit = None, -1

# Loop Evolutivo
for g in range(generations):
    fitnesses = np.array([calculate_fitness(ind) for ind in population])

    gen_best_idx = int(np.argmax(fitnesses))
    if fitnesses[gen_best_idx] > best_fit:
        best_fit = int(fitnesses[gen_best_idx])
        best_ind = population[gen_best_idx].copy()

    convergence_best.append(int(fitnesses[gen_best_idx]))
    convergence_mean.append(float(fitnesses.mean()))

    print(f"Geracao {g+1:2d} | Melhor fitness: {fitnesses[gen_best_idx]:2d} | "
          f"Media: {fitnesses.mean():5.2f} | Melhor individuo: {population[gen_best_idx]}")

    new_population = []
    for _ in range(pop_size // 2):
        p1 = tournament_selection(population, fitnesses)
        p2 = tournament_selection(population, fitnesses)
        c1, c2 = crossover(p1, p2)
        new_population.extend([mutate(c1), mutate(c2)])

    new_population[0] = best_ind.copy()   # Elitismo
    population = np.array(new_population)

print("\n[LAB 02 - SUCESSO] "
      f"Melhor Individuo: {best_ind} | "
      f"Itens: {[i for i, b in enumerate(best_ind) if b == 1]} | "
      f"Peso: {int(np.sum(best_ind * weights))}/{max_weight} | "
      f"Valor (fitness): {best_fit}")

# ==========================================================
# Convergencia em formato de tabela (saida textual)
# ==========================================================
print("\n[LAB 02] Convergencia por geracao:")
print(" Ger | Melhor | Media")
print("-----+--------+-------")
for g in range(generations):
    print(f" {g+1:3d} | {convergence_best[g]:6d} | {convergence_mean[g]:5.2f}")

print(f"\n[LAB 02] Fitness na 1a geracao: {convergence_best[0]} | "
      f"Fitness final: {convergence_best[-1]} | "
      f"Geracao em que o otimo foi atingido: {convergence_best.index(max(convergence_best)) + 1}")

Geracao  1 | Melhor fitness: 13 | Media:  5.10 | Melhor individuo: [0 1 1 1 0]
Geracao  2 | Melhor fitness: 14 | Media:  8.10 | Melhor individuo: [0 1 0 1 1]
Geracao  3 | Melhor fitness: 15 | Media:  6.50 | Melhor individuo: [0 1 1 1 1]
Geracao  4 | Melhor fitness: 15 | Media: 11.70 | Melhor individuo: [0 1 1 1 1]
Geracao  5 | Melhor fitness: 15 | Media: 11.80 | Melhor individuo: [0 1 1 1 1]
Geracao  6 | Melhor fitness: 15 | Media:  9.90 | Melhor individuo: [0 1 1 1 1]
Geracao  7 | Melhor fitness: 15 | Media: 11.60 | Melhor individuo: [0 1 1 1 1]
Geracao  8 | Melhor fitness: 15 | Media: 13.40 | Melhor individuo: [0 1 1 1 1]
Geracao  9 | Melhor fitness: 15 | Media: 13.10 | Melhor individuo: [0 1 1 1 1]
Geracao 10 | Melhor fitness: 15 | Media: 10.20 | Melhor individuo: [0 1 1 1 1]

[LAB 02 - SUCESSO] Melhor Individuo: [0 1 1 1 1] | Itens: [1, 2, 3, 4] | Peso: 8/15 | Valor (fitness): 15

[LAB 02] Convergencia por geracao:
 Ger | Melhor | Media
-----+--------+-------
   1 |     13 |  5.10
